# Scanpy expression plots with GenomeSpy

Compare marker expression across annotated cell groups using Scanpy’s packaged `pbmc68k_reduced()` dataset. Scanpy owns the analysis; `genome_spy.integrations.scanpy` renders supported plotting calls. This workflow adapts the [core plotting tutorial](https://scanpy.scverse.org/en/stable/tutorials/plotting/core.html). Install this project with its `scanpy` extra and select that environment as the notebook kernel. The preview targets Scanpy 1.11.5.

This workflow compares dotplot, matrixplot, individual-cell heatmap, supplied matrix and clustermap views; see the [roadmap](../PLAN.md) for remaining options. For embeddings, linked brushing and UMAP customization, use the [UMAP notebook](scanpy_umap.ipynb).

Compute the group dendrogram once in Python with `sc.tl.dendrogram`. It uses this dataset’s existing `X_pca` coordinates. All three plotting calls reuse the stored linkage and leaf order; GenomeSpy only renders the branches. UMAP and PCA use the supplied coordinates. A later preparation cell explicitly computes a separate complete Wilcoxon ranking with Scanpy for statistic colors.

The examples use `zoom=True` for y-only wheel zoom and drag panning, keeping gene columns fixed. Use `zoom="xy"` to enable both axes, or `zoom=False` to disable navigation.


In [ ]:
import scanpy as sc

import genome_spy.integrations.scanpy as gsc

pbmc = sc.datasets.pbmc68k_reduced()
sc.tl.dendrogram(pbmc, groupby="bulk_labels")
pbmc

## Dotplot: summarize marker expression

Use the core tutorial’s marker dictionary and the dataset’s existing `bulk_labels`. Dot color shows the mean expression per group; dot area shows the fraction of cells with expression strictly above zero. This call uses `raw.X` by default, as in Scanpy. Hover a dot to inspect its gene, group, mean expression, and expressing-cell percentage. Set `hover=False` to disable tooltips. Summaries are calculated in Python from selected genes; GenomeSpy renders the chart and its guides.

The dendrogram orders the cell groups by their existing PCA profiles. Marker blocks retain their original order because their labels differ from the cell-group labels. The plotting call reads and validates the stored tree without changing it.

`highlight=True` uses a native two-axis ruler to mark the row and column centers under the pointer, including gaps between dots. The crosshair clears when the pointer leaves the plot. No hover selection or extra background data is needed. `zoom=True` enables y-only wheel zoom and drag panning. The percentage-to-dot-size mapping stays fixed while navigating.

Marker blocks have four-pixel gaps. Gene labels, marker brackets and the dendrogram use aligned native side tracks; legends sit outside the complete chart. The tracks share the same gapped positions while zooming.


In [ ]:
marker_genes = {
    "B-cell": ["CD79A", "MS4A1"],
    "Dendritic": ["FCER1A", "CST3"],
    "Monocytes": ["FCGR3A"],
    "NK": ["GNLY", "NKG7"],
    "Other": ["IGLL1"],
    "Plasma": ["IGJ"],
    "T-cell": ["CD3D"],
}
dotplot = gsc.pl.dotplot(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    zoom=True,
    highlight=True,
    width=400,
    height=350,
    show=False,
)
dotplot.display(inline=True, controls=False)

## Matrixplot: compare scaled group means

Reuse the same PBMC data, marker dictionary and `bulk_labels`. This adapts the core tutorial’s matrixplot: Python computes each group’s mean expression, then `standard_scale="var"` scales each gene’s means from 0 to 1. Constant genes become zero. Tile color shows that scaled mean; it does not show the fraction of expressing cells.

Hover a tile to see its gene, group and scaled mean. The same stored dendrogram gives both plots identical cell-group order. No clustering or embedding is recomputed by the plotting call.

Use the wheel to zoom y and drag to pan y. `show_values=True` reveals the plotted scaled means to two decimals when each tile has enough room. Text contrast follows the native tile palette. Both options are off by default.

Matrixplot tiles stay adjacent across marker blocks. Gene labels, marker brackets and the dendrogram use aligned native side tracks; legends sit outside the complete chart. The tracks share the same positions while zooming.


In [ ]:
matrixplot = gsc.pl.matrixplot(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    zoom=True,
    show_values=True,
    standard_scale="var",
    cmap="Blues",
    colorbar_title="column scaled\nexpression",
    width=400,
    height=350,
    show=False,
)
matrixplot.display(inline=True, controls=False)

## Heatmap: inspect individual cells

Reuse the same markers and stored dendrogram. Each row represents one cell rather than a group mean. Cells follow their existing group labels and the stored tree order. Four-pixel gaps separate cell groups and marker blocks; the gaps keep their width while zooming. The colored `bulk_labels` band, centered group labels, gene brackets and tree each have their own aligned native track.

`standard_scale="var"` scales each gene’s individual-cell values from 0 to 1 before sorting. These differ from matrixplot’s scaled group means. Hover a tile to inspect its original cell ID, gene, group and scaled expression; hover the colored band to see its cell group. `hover=False` disables both. No cells are sampled or averaged, and no clustering is rerun.

Use the wheel to zoom y and drag to pan y; gene columns stay fixed. Set `zoom="xy"` to navigate both axes, or `swap_axes=True` to put cells on x. The tracks and group gaps follow the same cell and gene positions.

`show_values=True` reveals each plotted cell value once you zoom close enough. It uses the same fitted text as matrixplot; narrow cell rows remain uncluttered at the overview. Hover still works over the text.


In [ ]:
heatmap = gsc.pl.heatmap(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    standard_scale="var",
    cmap="viridis",
    width=400,
    height=450,
    zoom=True,
    show_values=True,
    show=False,
)
heatmap.display(inline=True, controls=False)

## A supplied numeric matrix

`matrix` displays values you already have; it does not read AnnData or compute summaries. Here it displays the group correlations already computed by `sc.tl.dendrogram` above, in the original category order. Unlike `correlation_matrix`, this generic helper does not attach a tree.

This chart also enables y-only wheel zoom, drag panning and fitted values with `zoom=True, show_values=True`. The text shows the supplied correlation, rounded to two decimals; hover retains more precision.

Row and column labels use native side tracks, sharing the tiles’ index scales while zooming and panning. The supplied row and column order stays unchanged.


In [ ]:
group_labels = pbmc.obs["bulk_labels"].cat.categories.tolist()
matrix_values = pbmc.uns["dendrogram_bulk_labels"]["correlation_matrix"]
matrix = gsc.pl.matrix(
    matrix_values,
    zoom=True,
    show_values=True,
    xticks=group_labels,
    yticks=group_labels,
    xlabel="Cell group",
    ylabel="Cell group",
    title="Stored group correlations",
    color_map="viridis",
    width=400,
    height=350,
    show=False,
)
matrix.display(inline=True, controls=False)

## Correlation matrix: reuse the stored group tree

Display the same stored correlations in dendrogram order, with native group labels and a left tree. This call reuses `sc.tl.dendrogram` results from the first cell; it does not recompute correlations or linkage. Labels and branches have their own aligned viewports. Vertical column labels keep long group names inside the top track.


In [ ]:
correlations = gsc.pl.correlation_matrix(
    pbmc,
    "bulk_labels",
    show_correlation_numbers=True,
    width=400,
    height=350,
    show=False,
)
correlations.display(inline=True, controls=False)

## Clustermap: cluster individual cells and genes

Clustermap compares the same marker genes while computing a separate tree for individual cells and another for genes. These are different from the group dendrogram used above. As in Scanpy’s clustermap, Python uses SciPy linkage with average linkage and Euclidean distance; GenomeSpy draws the resulting branches and reordered cells. Supplying `row_linkage` and `col_linkage` reuses your own trees instead.

Select the ten marker genes from the dataset’s existing log-expression snapshot before plotting. The row color strip keeps the saved `bulk_labels` colors while cells are reordered. Its native `bulk_labels` legend identifies the cell types using the same saved category order and colors; the expression colorbar remains separate. Hover the strip to inspect its cell group, or a tile to inspect the cell, gene and value. Both use the existing `hover` setting. Hide the 700 cell-name ticks with `yticklabels=False`; hover still identifies each cell, gene and expression value. No cells are sampled, and no embedding or gene ranking is rerun.


Use the wheel to zoom the cell rows and drag to pan. The gene columns stay fixed; set `zoom="xy"` to navigate both axes. Values appear when the tiles become large enough. Both trees and color strips follow the same navigation.

All dendrograms in this workflow use GenomeSpy’s native right-angle links through the shared renderer. Python still computes the trees; rendering keeps their stored merge heights and leaf order. While zooming or panning, the tree height scale fits branches contained in the visible range, expanding local detail without changing the computed heights.


In [ ]:
marker_names = [gene for genes in marker_genes.values() for gene in genes]
cluster_data = pbmc.raw[:, marker_names].to_adata()
clustermap = gsc.pl.clustermap(
    cluster_data,
    obs_keys="bulk_labels",
    use_raw=False,
    cmap="viridis",
    yticklabels=False,
    zoom=True,
    show_values=True,
    width=400,
    height=450,
    show=False,
)
clustermap.display(inline=True, controls=False)

## Stored gene rankings: inspect the existing scores

The packaged PBMC object already contains `uns["rank_genes_groups"]`, computed with Scanpy's logistic-regression method. Read those stored scores for B cells and monocytes; this plotting call never fits differential expression. For your own data, run `sc.tl.rank_genes_groups(...)` explicitly during analysis before plotting.

Gene labels sit at their stored rank and score. The two panels share score limits, and hovering a label shows its score and zero-based rank. `width` and `height` set native panel dimensions without Matplotlib.

In [ ]:
rank_key = "rank_genes_groups"
ranked_groups = ["CD19+ B", "CD14+ Monocyte"]
rank_scores = gsc.pl.rank_genes_groups(
    pbmc,
    groups=ranked_groups,
    n_genes=5,
    key=rank_key,
    ncols=2,
    width=240,
    height=300,
    show=False,
)
rank_scores.display(inline=True, controls=False)

## Ranked expression plots: reuse the same chart families

These calls use `sc.get.rank_genes_groups_df` to select three stored genes per requested rank list. `groups` selects marker lists; all 700 cells and all ten expression groups remain available. Repeated genes retain separate slots. No ranking is rerun, and the existing group dendrogram is reused.

The first chart shows mean expression and expressing-cell fractions. The next shows scaled group means, and the last shows scaled individual-cell expression. They reuse the same y zoom, fitted values, native ruler and tooltips as the earlier examples. Use `zoom="xy"` to navigate both axes.

The packaged logreg result has scores but no log-fold changes or p-values. `min_logfoldchange` therefore requires an explicitly prepared result that contains fold changes. The statistic-color examples below prepare a separate complete result explicitly.

In [ ]:
ranked_dotplot = gsc.pl.rank_genes_groups_dotplot(
    pbmc,
    groups=ranked_groups,
    n_genes=3,
    key=rank_key,
    zoom=True,
    highlight=True,
    width=400,
    height=350,
    show=False,
)
ranked_dotplot.display(inline=True, controls=False)

In [ ]:
ranked_matrixplot = gsc.pl.rank_genes_groups_matrixplot(
    pbmc,
    groups=ranked_groups,
    n_genes=3,
    key=rank_key,
    standard_scale="var",
    cmap="Blues",
    zoom=True,
    show_values=True,
    width=400,
    height=350,
    show=False,
)
ranked_matrixplot.display(inline=True, controls=False)

In [ ]:
ranked_heatmap = gsc.pl.rank_genes_groups_heatmap(
    pbmc,
    groups=ranked_groups,
    n_genes=3,
    key=rank_key,
    standard_scale="var",
    cmap="viridis",
    zoom=True,
    show_values=True,
    width=400,
    height=450,
    show=False,
)
ranked_heatmap.display(inline=True, controls=False)

## Color ranked summaries by stored statistics

Dot sizes can describe expression while colors describe differential-expression results. The packaged logreg cache has only 100 genes per group; Scanpy requires complete rankings for statistic colors. Prepare a separate complete Wilcoxon result explicitly with Scanpy, leaving the packaged rankings available for the earlier examples.

The plotting calls below only read this saved result. `groups` selects two marker lists; colors cover all ten expression groups. Missing statistics raise an error instead of triggering analysis or substituting expression values.


In [ ]:
sc.tl.rank_genes_groups(
    pbmc,
    "bulk_labels",
    method="wilcoxon",
    use_raw=True,
    n_genes=pbmc.n_vars,
    key_added="rank_stats",
)

### Dotplot: fold-change colors and expression fractions

Color shows stored log2 fold change, while dot size still shows the fraction of cells above the expression cutoff. Zoom and the native ruler work as before. `layer` or `use_raw` controls expression fractions independently of the stored ranking’s source.


In [ ]:
statistic_dotplot = gsc.pl.rank_genes_groups_dotplot(
    pbmc,
    groups=ranked_groups,
    n_genes=3,
    key="rank_stats",
    values_to_plot="logfoldchanges",
    cmap="redblue",
    vmin=-4,
    vmax=4,
    zoom=True,
    highlight=True,
    width=400,
    height=350,
    show=False,
)
statistic_dotplot.display(inline=True, controls=False)

### Matrixplot: adjusted p-values

`log10_pvals_adj` displays **−log10(adjusted p-value)**, so larger values indicate smaller p-values. Stored statistics keep their values without expression normalization. Zoom reveals fitted numeric labels; hover reports the displayed value. Ranked individual-cell heatmaps continue to show expression.


In [ ]:
statistic_matrixplot = gsc.pl.rank_genes_groups_matrixplot(
    pbmc,
    groups=ranked_groups,
    n_genes=3,
    key="rank_stats",
    values_to_plot="log10_pvals_adj",
    cmap="viridis",
    zoom=True,
    show_values=True,
    width=400,
    height=350,
    show=False,
)
statistic_matrixplot.display(inline=True, controls=False)

## Variable-gene diagnostics: inspect the stored selection

`highly_variable_genes` compares normalized and raw dispersion against mean expression. The packaged PBMC `var` table already contains these statistics and its selected-gene flag. Pass this table directly: the dataset predates Scanpy’s `uns["hvg"]` flavor metadata. The plotting call does not rerun gene selection or read expression. Black points are selected genes; gray points are other genes. Hover for gene names and values, and zoom or pan either plot independently.

In [ ]:
variable_genes = gsc.pl.highly_variable_genes(
    pbmc.var,
    width=280,
    height=240,
    zoom=True,
    show=False,
)
variable_genes.display(inline=True, controls=False)

## Highest-expression genes: check count percentages

Load Scanpy’s PBMC3k **counts** separately: the processed PBMC object above is
scaled and must not be used for count percentages. Scanpy downloads this dataset
on first use and caches it locally. No preprocessing is needed for this plot.

`highest_expr_genes` uses Scanpy total normalization to calculate each cell’s
gene percentages, then selects the genes with the largest mean percentage.
Boxes show the middle half of the cells; the line is the median, whiskers extend
to observations within 1.5 times the box width, and small points show outliers.
Hover for summaries or outlier cell IDs; scroll to zoom the percentage axis.


In [ ]:
counts = sc.datasets.pbmc3k()
highest_expression = gsc.pl.highest_expr_genes(
    counts, n_top=20, width=480, height=420, zoom=True, show=False
)
highest_expression.display(inline=True, controls=False)

## Scrublet score distributions: inspect stored doublet scores

Use the same PBMC3k counts. Run **Scanpy** explicitly to simulate doublets and
calculate scores, on a copy so the earlier count example remains unchanged.
The fixed seed makes the example reproducible. This demonstration supplies a
threshold of 0.25; in your analysis, choose and inspect your own threshold.

The GenomeSpy call only reads the saved results. It shows observed and simulated
scores in 49 fixed bins between 0 and 1, normalized separately to probability
density. The observed density uses a log axis by default; simulated density is
linear. The black line is the stored threshold. Hover for each populated bin's
count and density, or scroll to zoom a panel's score axis. Stored batch results
produce one pair of panels per batch, in category order.


In [ ]:
doublet_data = counts.copy()
sc.pp.scrublet(
    doublet_data,
    threshold=0.25,
    random_state=0,
    use_approx_neighbors=False,
    verbose=False,
)

In [ ]:
scrublet_scores = gsc.pl.scrublet_score_distribution(
    doublet_data, width=300, height=220, zoom=True, show=False
)
scrublet_scores.display(inline=True, controls=False)

## Ordered profiles: timeseries_as_heatmap

Plot supplied profiles without computing or changing their order. Here Scanpy's
public getter reads the same ten PBMC markers from the raw snapshot, and Python
explicitly sorts cells by their **stored S-phase score**. This is a score-ordered
expression view, not measured time or inferred pseudotime.

Variables run from bottom to top, following Scanpy's helper. The dashed rule
marks the midpoint of the supplied cell order. Hover reports variable, sample
position and expression. Scroll to zoom both axes, drag to pan, and double-click
to reset. Values appear when there is room for them.


In [ ]:
profiles = sc.get.obs_df(pbmc, keys=marker_names, use_raw=True)
profiles = profiles.loc[pbmc.obs["S_score"].sort_values(kind="stable").index]

In [ ]:
timeseries_heatmap = gsc.pl.timeseries_as_heatmap(
    profiles.to_numpy(),
    var_names=profiles.columns.to_numpy(),
    highlights_x=[(len(profiles) - 1) / 2],
    color_map="viridis",
    width=600,
    height=280,
    zoom="xy",
    show_values=True,
    show=False,
)
timeseries_heatmap.display(inline=True, controls=False)

## Time-series points: reuse the supplied profiles

Scanpy's `timeseries` draws **points**, not connecting lines. Reuse three of the
same raw-snapshot markers and their explicit S-score ordering from the preceding
heatmap. Pass the stored S-phase scores as the x coordinates; duplicate scores
stay at their supplied positions. This is a score-ordered view, not measured time.

The native legend names the marker genes. Hover reports series, supplied sample
position, score and expression. Scroll to zoom both axes, drag to pan and
double-click to reset. The dashed line marks score zero; it does not split or
recompute the data.


In [ ]:
time_series = gsc.pl.timeseries(
    profiles[["CD79A", "CD3D", "GNLY"]],
    time=pbmc.obs.loc[profiles.index, "S_score"].to_numpy(),
    var_names=["CD79A", "CD3D", "GNLY"],
    xlabel="Stored S-phase score",
    ylabel="Log expression (raw snapshot)",
    highlights_x=[0],
    marker="o",
    width=600,
    height=300,
    zoom=True,
    show=False,
)
time_series.display(inline=True, controls=False)

## Compose time-series subplots with native panels

`timeseries_subplot` has the same explicit plotting arguments as Scanpy, with a
native chart returned by `show=False`. Compose panels with `gs.hconcat` instead
of creating Matplotlib axes. Both panels reuse GNLY expression and the same
stored S-phase coordinates above; no scores or analysis are recomputed.

The left panel colors the 700 cells by their stored cell-cycle phase; categories
sort as G1, G2M, S. The right panel colors the same values by their stored G2M
score. As in Scanpy, numeric coloring has a series key rather than a colorbar.
Scales and legends are independent: hover either panel, scroll to zoom its x/y
axes, drag to pan, and double-click to reset that panel.


In [ ]:
import genome_spy as gs

subplot_options = dict(
    time=pbmc.obs.loc[profiles.index, "S_score"].to_numpy(),
    xlabel="Stored S-phase score",
    ylabel="GNLY (log expression)",
    marker="o",
    width=280,
    height=220,
    zoom=True,
    show=False,
)
phase_panel = gsc.pl.timeseries_subplot(
    profiles["GNLY"].to_numpy(),
    color=pbmc.obs.loc[profiles.index, "phase"].to_numpy(),
    var_names=sorted(pbmc.obs["phase"].unique()),
    **subplot_options,
)
score_panel = gsc.pl.timeseries_subplot(
    profiles["GNLY"].to_numpy(),
    color=pbmc.obs.loc[profiles.index, "G2M_score"].to_numpy(),
    var_names=["GNLY"],
    **subplot_options,
)
time_series_subplots = (
    gs.hconcat(phase_panel, score_panel, spacing=30)
    .resolve_scale(default="independent")
    .resolve_legend(default="independent")
)
time_series_subplots.display(inline=True, controls=False)

## Stored simulations: points and heatmap

Load Scanpy's packaged `krumsiek11` simulation: four concatenated realizations,
160 observations each. Select five regulatory genes for readable guides. This
loads existing data; no simulation runs here. `tmax_realization` marks the runs
without trimming or reordering observations.

Both calls reuse the time-series renderers. Hover shows the values; wheel zoom
and drag pan the plots. The heatmap reveals fitted values as cells gain room.
`shuffle=True` would use Scanpy's fixed seed-1 permutation and always draw points.

Scanpy's loader warns about repeated observation names across these runs.
The plots use row positions and retain all 640 observations; the gene subset
stays a read-only view.


In [ ]:
simulation = sc.datasets.krumsiek11()
sim_profiles = simulation[:, ["Gata2", "Gata1", "Cebpa", "Pu.1", "Gfi1"]]
simulation_points = gsc.pl.sim(
    sim_profiles,
    tmax_realization=160,
    marker="o",
    zoom=True,
    width=480,
    height=230,
    show=False,
)
simulation_points.display(inline=True, controls=False)

In [ ]:
simulation_heatmap = gsc.pl.sim(
    sim_profiles,
    tmax_realization=160,
    as_heatmap=True,
    zoom="xy",
    show_values=True,
    width=480,
    height=230,
    show=False,
)
simulation_heatmap.display(inline=True, controls=False)

## DPT groups and pseudotime: inspect stored ordering

Run the analysis on the full packaged Krumsiek11 data from above. Scanpy's neighbors,
diffusion map and DPT calls are explicit, with the dataset's supplied root
(`uns["iroot"] = 0`). One branching split supplies the group, order and boundary
metadata required by this plot. The analysis is illustrative on this simulation.

Plot every other observation as a read-only view. The stored DPT ranks and
branching boundaries keep their full-data coordinates; no analysis is rerun
and no ranks are renumbered.

The GenomeSpy call only reads those saved fields. The upper panel shows category
codes (not gene expression); the lower shows pseudotime, colored by its value.
Both use the stored order coordinates without sorting the supplied observations.
Dashed rules mark stored group boundaries. Hover reports the original row
position, order and value; zoom and pan operate independently in each panel.


In [ ]:
sc.pp.neighbors(simulation, n_neighbors=10, use_rep="X", random_state=0)
sc.tl.diffmap(simulation, random_state=0)
sc.tl.dpt(simulation, n_dcs=10, n_branchings=1)

In [ ]:
dpt_cells = simulation[::2]
dpt_groups_chart = gsc.pl.dpt_groups_pseudotime(
    dpt_cells,
    marker="o",
    zoom=True,
    width=480,
    height=180,
    show=False,
)
dpt_groups_chart.display(inline=True, controls=False)

## DPT expression profiles: reuse the stored row indices

Reuse the DPT analysis above and select the same five simulation genes while
retaining all 640 observations. `dpt_order_indices` selects rows of the supplied
X in saved DPT order; it differs from the `dpt_order` coordinates used by the
previous diagnostic. Gene-only slicing keeps those indices valid. Observation
filtering/reordering can leave stale row indices; plotting never rebuilds them.

The default heatmap reuses the existing time-series matrix renderer. Genes stay
in the supplied order, with the first at the bottom; dashed rules mark the stored
branching boundaries. Hover reports the gene, ordered sample position and value.
Both-axis zoom and fitted values use the shared matrix interactions. Set
`as_heatmap=False` for the existing point renderer instead; it draws scatter
points with a gene legend, without smoothing or connecting samples.


In [ ]:
dpt_profiles = simulation[:, sim_profiles.var_names]
dpt_expression = gsc.pl.dpt_timeseries(
    dpt_profiles,
    zoom="xy",
    show_values=True,
    width=600,
    height=240,
    show=False,
)
dpt_expression.display(inline=True, controls=False)

## PAGA: inspect stored group connectivity

Reuse the packaged PBMC neighbor graph and compute **PAGA with Scanpy** explicitly.
Nodes represent the existing `bulk_labels` groups, not individual cells. Node area
uses Scanpy's group sizes; edge width shows stored connectivity. Hover reports
groups, cell counts or edge weights, and wheel/drag navigates both axes.

When `pos` is omitted, the plotting adapter uses Scanpy's public `plot=False`
path to compute only the graph layout, then renders native GenomeSpy marks.
It caches the successful coordinates, as Scanpy does. Pass
`pos=pbmc.uns["paga"]["pos"]` to reuse them in another chart.

CSR/CSC graphs keep their stored edge order. LIL graphs use the same layout path
after SciPy prepares compatible sparse storage. Stored graphs stay unchanged.


In [ ]:
sc.tl.paga(pbmc, groups="bulk_labels")

In [ ]:
paga_graph = gsc.pl.paga(
    pbmc,
    threshold=0.03,
    fontsize=9,
    width=700,
    height=460,
    zoom=True,
    show=False,
)
paga_graph.display(inline=True, controls=False)

**Data use and provenance**

`scanpy.datasets.pbmc68k_reduced()` loads Scanpy’s packaged, processed subset of 10x Genomics PBMC68k. No dataset copy is bundled here. The expression plots read selected genes in Python using existing annotations. Dotplot and matrixplot calculate group means; dotplot also calculates expressing-cell fractions. Heatmap keeps all 700 individual cells. Matrixplot scales each gene’s group means; heatmap scales its cell values. Scanpy computes the shared group linkage explicitly from existing PCA coordinates; GenomeSpy renders its branches. The supplied matrix displays the same stored group correlations without recomputing them.
Clustermap uses the same ten raw-snapshot marker genes and all 700 cells. Python computes its cell and gene linkage with SciPy, as in Scanpy/Seaborn; GenomeSpy renders the supplied values and tree geometry.

The ranked examples read the packaged logistic-regression names and scores. The result getter only selects stored marker lists; ranked expression charts read those genes from the existing raw snapshot and reuse the same group summaries or cell heatmap.

The statistic-color examples explicitly compute complete Wilcoxon rankings with Scanpy on the packaged log-expression snapshot, under `rank_stats`. GenomeSpy only renders the stored results.


`scanpy.datasets.pbmc3k()` loads 2,700 cells × 32,738 genes of unnormalized
10x Genomics PBMC counts, as used in Scanpy’s PBMC3k tutorial. The highest-expression
example uses these counts directly; normalization is for plotting only. The
downloaded data stays in the local Scanpy cache and is not vendored here.

Scrublet is explicitly computed with Scanpy on a copy of the PBMC3k counts,
using seed 0 and a demonstration threshold of 0.25. GenomeSpy reads those stored
observed/simulated scores; it does not run doublet detection.

The ordered-profile heatmap uses the same raw-snapshot markers and all 700 PBMC
cells, sorted explicitly by the existing S-phase score. GenomeSpy does not
compute time, density, trajectories or new expression statistics for this view.

The time-series points reuse three of those markers and the same supplied cell
order, with the stored S-phase score as x. No lines, smoothing or analysis run.

The composed time-series panels reuse GNLY and the stored phase/G2M scores for
the same 700 cells, preserving the supplied positions and values.


`scanpy.datasets.krumsiek11()` supplies the packaged Krumsiek et al. (2011)
simulated myeloid regulatory profiles (640 observations, 11 genes); five genes
are shown in four stored realizations. No simulation is rerun.

The DPT example explicitly computes neighbors (10 neighbors, X representation),
diffusion coordinates (seed 0) and one DPT branching on the same 640-sample
Krumsiek11 data. GenomeSpy displays a filtered metadata view and gene-selected expression
profiles using the same stored analysis, without rerunning it.

PAGA uses the packaged PBMC neighbor graph and `bulk_labels`; Scanpy computes
group connectivity explicitly, and its public no-plot API supplies graph layout
coordinates. GenomeSpy renders the stored groups and weighted edges.
